# Chatbot for FAQs

In [6]:
!pip install gradio nltk scikit-learn -q

In [7]:
import nltk
import re
import gradio as gr
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
nltk.download('punkt')
nltk.download('stopwords')

print("✅ NLTK resources downloaded")

✅ NLTK resources downloaded


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
faqs = [

    {
        "questions": [
            "What are the admission requirements?",
            "Admission criteria",
            "Eligibility for AI program",
            "How can I apply for AI degree?"
        ],

        "answer":
        "To apply for the AI program, you need FSc Pre-Engineering or equivalent with at least 60% marks."
    },

    {
        "questions": [
            "When do admissions start?",
            "Admission opening date",
            "Are admissions open?"
        ],

        "answer":
        "Admissions usually open in July for Fall admissions and December for Spring admissions."
    },

    {
        "questions": [
            "What is the fee structure?",
            "Semester fee",
            "Fee per semester"
        ],

        "answer":
        "The semester fee is approximately PKR 35,000–50,000."
    }

]

print("✅ FAQ dataset loaded")

✅ FAQ dataset loaded


In [10]:
stop_words = set(stopwords.words('english'))

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 1
    ]

    return " ".join(tokens)

print("✅ Preprocessing function ready")

✅ Preprocessing function ready


In [11]:
faq_questions = []
faq_answers = []

for faq in faqs:

    for question in faq["questions"]:

        faq_questions.append(preprocess(question))
        faq_answers.append(faq["answer"])

print("✅ Total Questions:", len(faq_questions))

✅ Total Questions: 10


In [12]:
vectorizer = TfidfVectorizer()

faq_vectors = vectorizer.fit_transform(faq_questions)

print("✅ TF-IDF vectorization complete")

✅ TF-IDF vectorization complete


In [13]:
def chatbot(user_question, threshold=0.35):

    clean_question = preprocess(user_question)

    if not clean_question.strip():

        return "Please enter a valid question.", 0

    user_vector = vectorizer.transform([clean_question])

    similarity_scores = cosine_similarity(
        user_vector,
        faq_vectors
    )

    best_index = np.argmax(similarity_scores)

    best_score = similarity_scores[0][best_index]

    if best_score < threshold:

        return (
            "❌ Sorry, I could not understand your question."
        ), best_score

    return faq_answers[best_index], best_score

print("✅ Chatbot function ready")

✅ Chatbot function ready


In [14]:
questions = [

    "when admissions start",
    "fee per semester",
    "admission criteria"
]

for q in questions:

    answer, score = chatbot(q)

    print("\nQuestion:", q)
    print("Answer:", answer)
    print("Score:", round(float(score), 2))


Question: when admissions start
Answer: Admissions usually open in July for Fall admissions and December for Spring admissions.
Score: 1.0

Question: fee per semester
Answer: The semester fee is approximately PKR 35,000–50,000.
Score: 1.0

Question: admission criteria
Answer: To apply for the AI program, you need FSc Pre-Engineering or equivalent with at least 60% marks.
Score: 1.0


In [15]:
def respond(message, history):

    answer, score = chatbot(message)

    response = f"{answer}\n\nSimilarity Score: {score:.2f}"

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    return history, ""


examples = [

    "When do admissions start?",
    "What is the fee structure?",
    "What are admission requirements?"
]

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🤖 AI FAQ Chatbot")

    chatbot_ui = gr.Chatbot(
        type="messages",
        height=450
    )

    with gr.Row():

        msg = gr.Textbox(
            placeholder="Ask your question...",
            scale=5
        )

        send = gr.Button(
            "Send 🚀",
            scale=1
        )

    gr.Examples(
        examples=examples,
        inputs=msg
    )

    send.click(
        respond,
        inputs=[msg, chatbot_ui],
        outputs=[chatbot_ui, msg]
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot_ui],
        outputs=[chatbot_ui, msg]
    )

demo.launch(share=True)

/tmp/ipykernel_2039/3030032413.py:27: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_2039/3030032413.py:31: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://141d0dfa217cd4e598.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
